# Analisi Multi-Risoluzione (MRA) - Data Fusion per Scala

Questo notebook implementa un'analisi multi-risoluzione per rilevare danni a diverse scale:

- **R (Rosso)** = H1 + V1 + D1 → Dettagli **FINI** (crepe piccole, alte frequenze) - da `coeffs[3]`
- **G (Verde)** = H2 + V2 + D2 → Dettagli **MEDI** (danni medi, medie frequenze) - da `coeffs[2]`
- **B (Blu)** = H3 + V3 + D3 → Dettagli **GROSSI** (crolli, basse frequenze) - da `coeffs[1]`

**IMPORTANTE**: `pywt.mra2()` restituisce i coefficienti in ordine DECRESCENTE di dettaglio (dal grossolano al fine).

## 1. Import e Setup

In [ ]:
import numpy as np
import pywt
import matplotlib.pyplot as plt
from pathlib import Path
import warnings

# Configurazione matplotlib
plt.rcParams['figure.figsize'] = (15, 10)
plt.rcParams['figure.dpi'] = 100

print(f"PyWavelets version: {pywt.__version__}")
print(f"NumPy version: {np.__version__}")

## 2. Configurazione Path

In [ ]:
# Per Google Colab, monta Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✓ Google Drive montato")
except:
    print("Non sei in Google Colab, usa path locali")

# Configura i path
DATA_FOLDER = '/content/drive/MyDrive/Tesi_magistrale/Data/'
OUTPUT_FOLDER = '/content/drive/MyDrive/Tesi_magistrale/Data/mra_results/'

# Crea cartella output se non esiste
Path(OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)

print(f"Data folder: {DATA_FOLDER}")
print(f"Output folder: {OUTPUT_FOLDER}")

## 3. Funzioni Principali

In [ ]:
def normalize_to_uint8(data):
    """Normalizza un array nel range [0, 255]."""
    data_min = np.min(data)
    data_max = np.max(data)
    
    if data_max - data_min > 1e-10:
        normalized = (data - data_min) / (data_max - data_min)
    else:
        normalized = np.zeros_like(data)
    
    return (normalized * 255).astype(np.uint8)


def apply_mra_fusion(image, wavelet='db4', level=3):
    """
    Applica la decomposizione MRA e crea l'immagine RGB fusa.
    
    Parameters
    ----------
    image : ndarray
        Immagine in input (2D)
    wavelet : str
        Tipo di wavelet (default: 'db4')
    level : int
        Numero di livelli (default: 3)
        
    Returns
    -------
    rgb_fused : ndarray
        Immagine RGB con data fusion (H, W, 3)
    mra_coeffs : list
        Coefficienti MRA
    detail_maps : dict
        Mappe di dettaglio per ogni scala
    """
    # Converti in 2D se necessario
    if image.ndim > 2:
        if image.shape[-1] == 3:
            image = np.mean(image, axis=-1)
        else:
            image = image.squeeze()
    
    # Applica MRA 2D
    print(f"Applicando MRA con wavelet '{wavelet}' e {level} livelli...")
    mra_coeffs = pywt.mra2(image, wavelet=wavelet, level=level,
                           transform='swt2', mode='periodization')
    
    print(f"Decomposizione completata. Livelli: {len(mra_coeffs)}")
    
    # Inizializza canali RGB
    height, width = image.shape
    R_channel = np.zeros((height, width), dtype=np.float64)
    G_channel = np.zeros((height, width), dtype=np.float64)
    B_channel = np.zeros((height, width), dtype=np.float64)
    
    # Dizionario per dettagli
    # IMPORTANTE: mra2() restituisce coefficienti in ordine DECRESCENTE
    # coeffs[1] = dettagli livello 3 (H3+V3+D3) - GROSSOLANI (basse freq)
    # coeffs[2] = dettagli livello 2 (H2+V2+D2) - MEDI (medie freq)
    # coeffs[3] = dettagli livello 1 (H1+V1+D1) - FINI (alte freq)
    detail_maps = {
        'fine': {'H': None, 'V': None, 'D': None, 'sum': None},       # coeffs[3]
        'medium': {'H': None, 'V': None, 'D': None, 'sum': None},     # coeffs[2]
        'coarse': {'H': None, 'V': None, 'D': None, 'sum': None}      # coeffs[1]
    }
    
    # Estrai dettagli (coeffs[1]=grosso, coeffs[2]=medio, coeffs[3]=fine)
    for i in range(1, min(4, len(mra_coeffs))):
        H, V, D = mra_coeffs[i]
        detail_sum = H + V + D
        
        if i == 1:
            # coeffs[1] = livello 3 = H3+V3+D3 → Canale B (GROSSO)
            B_channel = detail_sum
            detail_maps['coarse']['H'] = H
            detail_maps['coarse']['V'] = V
            detail_maps['coarse']['D'] = D
            detail_maps['coarse']['sum'] = detail_sum
            print(f"  B (H3+V3+D3, grosso): range [{B_channel.min():.3f}, {B_channel.max():.3f}]")
        elif i == 2:
            # coeffs[2] = livello 2 = H2+V2+D2 → Canale G (MEDIO)
            G_channel = detail_sum
            detail_maps['medium']['H'] = H
            detail_maps['medium']['V'] = V
            detail_maps['medium']['D'] = D
            detail_maps['medium']['sum'] = detail_sum
            print(f"  G (H2+V2+D2, medio):  range [{G_channel.min():.3f}, {G_channel.max():.3f}]")
        elif i == 3:
            # coeffs[3] = livello 1 = H1+V1+D1 → Canale R (FINE)
            R_channel = detail_sum
            detail_maps['fine']['H'] = H
            detail_maps['fine']['V'] = V
            detail_maps['fine']['D'] = D
            detail_maps['fine']['sum'] = detail_sum
            print(f"  R (H1+V1+D1, fine):   range [{R_channel.min():.3f}, {R_channel.max():.3f}]")
    
    # Normalizza e crea RGB
    R_norm = normalize_to_uint8(np.abs(R_channel))
    G_norm = normalize_to_uint8(np.abs(G_channel))
    B_norm = normalize_to_uint8(np.abs(B_channel))
    
    rgb_fused = np.stack([R_norm, G_norm, B_norm], axis=-1)
    
    print(f"Immagine RGB fusa: shape = {rgb_fused.shape}")
    
    return rgb_fused, mra_coeffs, detail_maps


print("✓ Funzioni caricate")

## 4. Carica Dataset

In [ ]:
# Carica il dataset
data_path = Path(DATA_FOLDER) / 'X_train_augmented.npy'

try:
    X = np.load(data_path)
    print(f"✓ Dataset caricato: {data_path}")
    print(f"  Shape: {X.shape}")
    print(f"  Dtype: {X.dtype}")
    print(f"  Range: [{X.min()}, {X.max()}]")
except FileNotFoundError:
    print(f"⚠ File non trovato: {data_path}")
    print("  Uso dati di esempio...")
    from pywt import data
    X = np.array([data.camera() for _ in range(5)])
    print(f"  Dati di esempio creati: shape = {X.shape}")

## 5. Parametri Analisi

In [ ]:
# Configura parametri
WAVELET = 'db4'        # Wavelet: 'db4', 'sym4', 'coif3', 'bior3.5'
LEVEL = 3              # Livelli: 2, 3, 4
NUM_SAMPLES = 5        # Quante immagini processare

print(f"Parametri configurati:")
print(f"  Wavelet: {WAVELET}")
print(f"  Livelli: {LEVEL}")
print(f"  Campioni: {NUM_SAMPLES}")

## 6. Processa Singola Immagine (Test)

In [ ]:
# Seleziona una immagine di test
test_idx = 0
test_image = X[test_idx]

print(f"Processando immagine {test_idx}...")
print(f"Shape: {test_image.shape}")

# Applica MRA fusion
rgb_fused, mra_coeffs, detail_maps = apply_mra_fusion(
    test_image,
    wavelet=WAVELET,
    level=LEVEL
)

print("\n✓ Analisi completata!")

## 7. Visualizza Risultati

In [ ]:
# Visualizza risultato principale
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

# Originale
axes[0].imshow(test_image, cmap='gray')
axes[0].set_title('Immagine Originale', fontsize=12, fontweight='bold')
axes[0].axis('off')

# RGB fuso
axes[1].imshow(rgb_fused)
axes[1].set_title('RGB Fuso\n(R=Fine, G=Medio, B=Grosso)', fontsize=12, fontweight='bold')
axes[1].axis('off')

# Canali separati
axes[2].imshow(rgb_fused[:, :, 0], cmap='Reds')
axes[2].set_title('R: Dettagli FINI\n(Crepe piccole)', fontsize=11, fontweight='bold', color='red')
axes[2].axis('off')

axes[3].imshow(rgb_fused[:, :, 1], cmap='Greens')
axes[3].set_title('G: Dettagli MEDI\n(Danni medi)', fontsize=11, fontweight='bold', color='green')
axes[3].axis('off')

axes[4].imshow(rgb_fused[:, :, 2], cmap='Blues')
axes[4].set_title('B: Dettagli GROSSI\n(Crolli)', fontsize=11, fontweight='bold', color='blue')
axes[4].axis('off')

plt.suptitle('Analisi Multi-Risoluzione (MRA) - Data Fusion', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Visualizza Dettagli per Livello

In [ ]:
# Visualizza dettagli H, V, D per ogni livello
fig, axes = plt.subplots(3, 4, figsize=(16, 12))

# ORDINE CORRETTO: coeffs[3]=fine(R), coeffs[2]=medio(G), coeffs[1]=grosso(B)
levels = ['fine', 'medium', 'coarse']
level_names = ['FINE - H1+V1+D1 (R)', 'MEDIO - H2+V2+D2 (G)', 'GROSSO - H3+V3+D3 (B)']

for row, (level, level_name) in enumerate(zip(levels, level_names)):
    if detail_maps[level]['H'] is not None:
        # H - Orizzontale
        axes[row, 0].imshow(np.abs(detail_maps[level]['H']), cmap='gray')
        axes[row, 0].set_title(f'{level_name}\nH (Orizzontale)', fontsize=10)
        axes[row, 0].axis('off')
        
        # V - Verticale
        axes[row, 1].imshow(np.abs(detail_maps[level]['V']), cmap='gray')
        axes[row, 1].set_title(f'{level_name}\nV (Verticale)', fontsize=10)
        axes[row, 1].axis('off')
        
        # D - Diagonale
        axes[row, 2].imshow(np.abs(detail_maps[level]['D']), cmap='gray')
        axes[row, 2].set_title(f'{level_name}\nD (Diagonale)', fontsize=10)
        axes[row, 2].axis('off')
        
        # Somma H+V+D
        axes[row, 3].imshow(np.abs(detail_maps[level]['sum']), cmap='hot')
        axes[row, 3].set_title(f'{level_name}\nH+V+D (Somma)', fontsize=10, fontweight='bold')
        axes[row, 3].axis('off')

plt.suptitle('Dettagli Wavelet per Livello (H, V, D)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Statistiche dei Dettagli

In [ ]:
# Calcola statistiche per ogni livello
print("Statistiche dei dettagli:\n")
print(f"{'Livello':<15} {'Mean':<12} {'Std':<12} {'Min':<12} {'Max':<12}")
print("="*63)

for level in levels:
    if detail_maps[level]['sum'] is not None:
        data = detail_maps[level]['sum']
        print(f"{level:<15} {data.mean():<12.4f} {data.std():<12.4f} "
              f"{data.min():<12.4f} {data.max():<12.4f}")

print("\nCanali RGB:")
print(f"{'Canale':<15} {'Mean':<12} {'Std':<12} {'Min':<12} {'Max':<12}")
print("="*63)
print(f"{'R (Fine)':<15} {rgb_fused[:,:,0].mean():<12.2f} {rgb_fused[:,:,0].std():<12.2f} "
      f"{rgb_fused[:,:,0].min():<12} {rgb_fused[:,:,0].max():<12}")
print(f"{'G (Medio)':<15} {rgb_fused[:,:,1].mean():<12.2f} {rgb_fused[:,:,1].std():<12.2f} "
      f"{rgb_fused[:,:,1].min():<12} {rgb_fused[:,:,1].max():<12}")
print(f"{'B (Grosso)':<15} {rgb_fused[:,:,2].mean():<12.2f} {rgb_fused[:,:,2].std():<12.2f} "
      f"{rgb_fused[:,:,2].min():<12} {rgb_fused[:,:,2].max():<12}")

## 10. Processa Multiple Immagini

In [ ]:
# Processa più immagini
num_to_process = min(NUM_SAMPLES, len(X))
all_rgb_fused = []

print(f"Processando {num_to_process} immagini...\n")

for idx in range(num_to_process):
    print(f"[{idx+1}/{num_to_process}] Immagine {idx}...", end=' ')
    
    image = X[idx]
    rgb_fused, _, _ = apply_mra_fusion(image, wavelet=WAVELET, level=LEVEL)
    all_rgb_fused.append(rgb_fused)
    
    # Salva immagine RGB fusa
    output_path = Path(OUTPUT_FOLDER) / f'rgb_fused_image_{idx:04d}.png'
    plt.imsave(output_path, rgb_fused)
    print(f"✓ Salvata")

# Converti in array
all_rgb_fused = np.array(all_rgb_fused)
print(f"\n✓ Processate tutte le immagini")
print(f"  Shape dataset RGB fuso: {all_rgb_fused.shape}")

## 11. Salva Risultati

In [ ]:
# Salva array completo
output_array_path = Path(OUTPUT_FOLDER) / 'all_rgb_fused.npy'
np.save(output_array_path, all_rgb_fused)

print(f"✓ Dataset RGB fuso salvato in:")
print(f"  {output_array_path}")
print(f"  Shape: {all_rgb_fused.shape}")
print(f"  Size: {all_rgb_fused.nbytes / (1024**2):.2f} MB")

## 12. Visualizza Griglia Risultati

In [ ]:
# Mostra griglia con tutte le immagini processate
n_images = len(all_rgb_fused)
cols = min(5, n_images)
rows = (n_images + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(cols*3, rows*3))
if rows == 1:
    axes = axes.reshape(1, -1)
if cols == 1:
    axes = axes.reshape(-1, 1)

for idx in range(n_images):
    row = idx // cols
    col = idx % cols
    axes[row, col].imshow(all_rgb_fused[idx])
    axes[row, col].set_title(f'Immagine {idx}', fontsize=10)
    axes[row, col].axis('off')

# Nascondi assi extra
for idx in range(n_images, rows * cols):
    row = idx // cols
    col = idx % cols
    axes[row, col].axis('off')

plt.suptitle('Dataset RGB Fuso - Tutte le Immagini', fontsize=14, fontweight='bold')
plt.tight_layout()

# Salva griglia
grid_path = Path(OUTPUT_FOLDER) / 'all_images_grid.png'
plt.savefig(grid_path, dpi=150, bbox_inches='tight')
print(f"✓ Griglia salvata in: {grid_path}")
plt.show()

## 13. Analisi Finale

### Interpretazione dei risultati:

- **Canale R (Rosso)**: Evidenzia dettagli fini (alte frequenze)
  - Utile per rilevare: crepe sottili, texture fine
  
- **Canale G (Verde)**: Evidenzia dettagli medi (medie frequenze)
  - Utile per rilevare: danni di dimensioni medie, bordi
  
- **Canale B (Blu)**: Evidenzia dettagli grossi (basse frequenze)
  - Utile per rilevare: grandi variazioni, crolli, strutture ampie

### Prossimi passi:

1. Usa `all_rgb_fused.npy` per training di modelli di deep learning
2. Applica thresholding per creare maschere di danno
3. Combina con altre features per migliorare la classificazione

In [ ]:
print("\n" + "="*80)
print("ANALISI COMPLETATA CON SUCCESSO!")
print("="*80)
print(f"\nRisultati salvati in: {OUTPUT_FOLDER}")
print(f"\nFile generati:")
print(f"  - all_rgb_fused.npy ({all_rgb_fused.shape})")
print(f"  - {num_to_process} immagini RGB fuse individuali")
print(f"  - all_images_grid.png")
print("\n✓ Pronto per il prossimo step della tua analisi!")